In [1]:
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from swmm_api import SwmmInput, swmm5_run
from swmm_api.input_file.section_labels import *
from swmm_api.input_file.sections import *

/home/javkt/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
nodes = gpd.read_file('nodes.geojson')
edges = gpd.read_file('edges.geojson')
subcatchments = gpd.read_file('subcatchments.geojson')

In [3]:
nodes['SWMM_ID'] = [f"JN_{i}" if t == 'junction' else f"OF_{i}" if t=='outfall' else f"IN_{i}" for i, t in enumerate(nodes['node_type'])]
subcatchments['SWMM_ID'] = [f"SUB_{i}" for i in range(len(subcatchments))]
edges['SWMM_ID'] = [f"CON_{i}" for i in range(len(edges))]

node_to_id = {row.node_id: row.SWMM_ID for _, row in nodes.iterrows()}

In [4]:
inp = SwmmInput()

inp[OPTIONS] = OptionSection(
    FLOW_UNITS='CMS',
    INFILTRATION='GREEN_AMPT',
    FLOW_ROUTING='DYNWAVE',
    START_DATE='01/01/2025',
    START_TIME='00:00:00',
    END_DATE='01/02/2025',
    END_TIME='00:00:00',
)

In [5]:
import numpy as np
from scipy.optimize import brentq

def _shape_function(t, peak):
    return (1 - t) * np.exp(-(np.log(t+1e-3) - peak)**2)

def _max_intensity_for_peak(peak, duration, total_depth):
    dt_hr = 15 / 60
    T = np.arange(0, duration, dt_hr)
    t = T / duration

    shape = _shape_function(t, peak)
    shape /= np.trapz(shape, t)

    # convert shape to intensities (mm/hr)
    intensity = shape * (total_depth / len(T))

    return np.max(intensity)

def generate_hyetograph_15m(max_intensity, duration, total_depth):
    # max_intensity: mm/hr
    max_intensity /= 4 # force the 15m rainfall not to exceed the average intensity of the hour
    def objective(peak):
        return _max_intensity_for_peak(peak, duration, total_depth) - max_intensity

    peak_solution = brentq(objective, -8, 0)

    dt_hr = 15 / 60
    T = np.arange(0, duration, dt_hr)
    t = T / duration
    t[0] = 1e-6

    shape = _shape_function(t, peak_solution)
    shape /= np.trapz(shape, t)
    depths = np.cumsum(shape * total_depth / len(T))

    data = list(zip(T, depths))

    gage = RainGage(
        name='RG',
        form='CUMULATIVE',
        interval='0:15:00',
        source='TIMESERIES',
        timeseries='TS',
        SCF=1.0,
        units='MM'
    )

    timeseries = TimeseriesData(
        name='TS',
        data=data
    )
    return gage, timeseries


max_intensity $\times$ duration > 2 $\cdot$ total_depth

(mm/hr) * (hr) > mm

In [ ]:
inp[RAINGAGES]['RG'], inp[TIMESERIES]['TS'] = generate_hyetograph_15m(100, 1, 25)

for i, row in subcatchments.iterrows():
    area = row.geometry.area
    width = area ** 0.5
    area /= 10_000 # m^2 to hectares
    inp[SUBCATCHMENTS][row.SWMM_ID] = SubCatchment(
        name=row.SWMM_ID,
        rain_gage='RG',
        outlet=node_to_id[row.node_id],
        area=area,
        width=width,
        slope=row.slope,
        imperviousness=row.pct_impervious
    )

    inp[INFILTRATION][row.SWMM_ID] = InfiltrationGreenAmpt(
        subcatchment=row.SWMM_ID,
        suction_head=row['Suction'], # (mm)
        hydraulic_conductivity=row['Ksat'], # (mm/hr)
        moisture_deficit_init=row['IMD']
    )

    # default runoff parameters
    inp[SUBAREAS][row.SWMM_ID] = SubArea(
        subcatchment=row.SWMM_ID,
        n_imperv=0.015,
        n_perv=0.15,
        storage_imperv=2.0, # (mm)
        storage_perv=5.0, # (mm)
        pct_zero=25.0
    )

In [8]:
for i, row in nodes.iterrows():
    if row['node_type'] == 'outfall':
        inp[OUTFALLS][row.SWMM_ID] = Outfall(
            name=row.SWMM_ID,
            elevation=row['invert_elevation'],
            kind = 'FREE'
        )
    else:
        inp[JUNCTIONS][row.SWMM_ID] = (Junction(
            name=row.SWMM_ID,
            elevation=row['invert_elevation']
        ))

In [9]:
for i, row in edges.iterrows():
    inp[CONDUITS][row.SWMM_ID] = Conduit(
        name=row.SWMM_ID,
        from_node=node_to_id[row.start_node],
        to_node=node_to_id[row.end_node],
        length=row.geometry.length,
        roughness=0.013, # Manning's n, mostly RCP AND PVC Pipes.
    )

    inp[XSECTIONS][row.SWMM_ID] = CrossSection(
        link=row.SWMM_ID,
        shape='CIRCULAR',
        height=row['diameter']
    )

In [10]:
inp.to_file('watershed_model.inp')

'watershed_model.inp'

In [11]:
swmm5_run('watershed_model.inp')


 o  Retrieving project data
